# Bước 2 — Dịch sách Hán → Việt bằng LLM

**Input:** các file `.txt` của bước 1 (*Add Input* → Output của notebook OCR, hoặc dataset chứa `.txt`).
**Output:** `/kaggle/working/dich/<tên sách>.txt` (bản dịch) và `dich.zip`.

**Settings:** Accelerator **GPU T4 x2** (2 × 16 GB), Internet **On** → sửa ô cấu hình → *Run All*.
Model nào trên Hugging Face có bản chat/instruct và vừa VRAM đều dùng được (xem `README.md`).

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
GPU_PER_MODEL = 2       # 1: model <= 4B, mỗi GPU chạy 1 bản (nhanh gấp đôi) | 2: model 7-8B trải trên 2 GPU
LOAD_4BIT = False       # True: nén 4-bit để chạy model ~14B
BATCH = 8               # số đoạn dịch cùng lúc; hết VRAM (OOM) thì giảm
CHUNK_CHARS = 300       # số chữ Hán tối đa mỗi đoạn đưa vào LLM
MAX_NEW_TOKENS = 1200
TEMPERATURE = 0         # 0 = luôn chọn từ chắc nhất; 0.3-0.7 = dịch đa dạng hơn
REPETITION_PENALTY = 1.05
LIMIT = None            # vd 20 = chỉ dịch 20 đoạn để thử
BUDGET_H = 10.5         # Kaggle cắt phiên ở 12 giờ

SYSTEM = "Bạn là dịch giả chuyên dịch sách Y học cổ truyền từ chữ Hán (văn ngôn và bạch thoại) sang tiếng Việt."
USER = ("Văn bản chữ Hán:\n{text}\n\n"
        "Dịch toàn bộ văn bản trên sang tiếng Việt, đầy đủ ý, văn phong tự nhiên. Tên huyệt, vị thuốc, bài thuốc, "
        "bệnh danh giữ theo âm Hán-Việt (vd: 足三里 -> Túc tam lý). Chỉ viết tiếng Việt: không chép lại chữ Hán, "
        "không dịch sang tiếng Anh, không lời dẫn, không giải thích.")

## 0. Cài đặt + code dịch (không cần sửa)

In [ ]:
import subprocess, sys
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(">", cmd[:150]); print("\n".join((r.stdout + r.stderr).strip().splitlines()[-3:]))

sh(f'{sys.executable} -m pip install -q -U transformers accelerate' + (" bitsandbytes" if LOAD_4BIT else ""))
try:  # model cần đăng nhập (Llama, Gemma...): Add-ons -> Secrets -> HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    import os; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import snapshot_download
snapshot_download(MODEL, ignore_patterns=["*.gguf", "*.pth", "*.onnx", "original/*"])

In [ ]:
%%writefile /kaggle/working/dich_worker.py
"""Dịch các đoạn Hán -> Việt bằng 1 LLM (transformers); chạy lại thì bỏ qua đoạn đã dịch."""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path


def load_done(out: Path) -> set[str]:
    done = set()
    for f in out.glob("dich_shard*.jsonl"):
        for line in open(f, encoding="utf-8"):
            try:
                done.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                pass
    return done


def build_prompt(tok, system: str, user: str) -> str:
    """Prompt theo chat template của model (model không nhận vai system thì gộp vào user)."""
    for msgs in ([{"role": "system", "content": system}, {"role": "user", "content": user}],
                 [{"role": "user", "content": system + "\n\n" + user}]):
        try:
            return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                           enable_thinking=False)  # Qwen3: tắt chế độ suy nghĩ
        except Exception:  # noqa: BLE001
            continue
    raise RuntimeError("Model không có chat template -- hãy chọn bản *-Instruct / *-Chat")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--chunks", required=True, help='jsonl, mỗi dòng {"id", "zh"}')
    ap.add_argument("--prompt", required=True, help='json {"system", "user"}; user chứa {text}')
    ap.add_argument("--out", required=True)
    ap.add_argument("--shard", type=int, default=0)
    ap.add_argument("--nshard", type=int, default=1)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--max-new", type=int, default=1200)
    ap.add_argument("--temperature", type=float, default=0.0, help="0 = luôn chọn từ chắc nhất")
    ap.add_argument("--rep-penalty", type=float, default=1.05)
    ap.add_argument("--load-4bit", action="store_true")
    ap.add_argument("--limit", type=int, default=None, help="chỉ dịch N đoạn (chạy thử)")
    a = ap.parse_args()
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)

    chunks = [json.loads(l) for l in open(a.chunks, encoding="utf-8") if l.strip()]
    done = load_done(out)
    todo = [c for i, c in enumerate(chunks) if i % a.nshard == a.shard and c["id"] not in done][:a.limit]
    print(f"[dich#{a.shard}] còn {len(todo)} đoạn", flush=True)
    if not todo:
        return

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(a.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = {"dtype": torch.float32}
    if torch.cuda.is_available():
        kw = {"dtype": torch.float16, "device_map": "auto"}  # T4 không có bf16
        if a.load_4bit:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(a.model, **kw).eval()
    pr = json.loads(Path(a.prompt).read_text(encoding="utf-8"))

    todo.sort(key=lambda c: len(c["zh"]), reverse=True)  # đoạn dài gần nhau -> ít padding
    t_start, n_done = time.time(), 0
    fout = open(out / f"dich_shard{a.shard}.jsonl", "a", encoding="utf-8")
    for b in range(0, len(todo), a.batch):
        batch = todo[b:b + a.batch]
        prompts = [build_prompt(tok, pr["system"], pr["user"].format(text=c["zh"])) for c in batch]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            sample = {"do_sample": True, "temperature": a.temperature} if a.temperature > 0 else {"do_sample": False}
            gen = model.generate(**enc, max_new_tokens=a.max_new, repetition_penalty=a.rep_penalty,
                                 pad_token_id=tok.pad_token_id, **sample)
        gen = gen[:, enc["input_ids"].shape[1]:]
        for c, g in zip(batch, gen):
            n_tok = int((g != tok.pad_token_id).sum())
            vi = tok.decode(g, skip_special_tokens=True).strip()
            fout.write(json.dumps({"id": c["id"], "vi": vi, "hit_max": n_tok >= a.max_new},
                                  ensure_ascii=False) + "\n")
        fout.flush()
        n_done += len(batch)
        eta = (time.time() - t_start) / n_done * (len(todo) - n_done) / 60
        print(f"[dich#{a.shard}] {n_done}/{len(todo)} đoạn | còn ~{eta:.0f} phút\n"
              f"   ZH: {batch[-1]['zh'][:60]}\n   VI: {vi[:120]}", flush=True)
    fout.close()


if __name__ == "__main__":
    main()

## 1. Đọc txt → chia đoạn

In [ ]:
import glob, hashlib, json, os, re, shutil, time
from pathlib import Path

N_GPU = len(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip().splitlines())
if N_GPU == 0:
    raise SystemExit("!!! Không có GPU: Settings -> Accelerator -> GPU T4 x2 rồi Run All lại.")
WORK = Path("/kaggle/working"); RAW = WORK / "dich_raw"; DICH = WORK / "dich"
RAW.mkdir(exist_ok=True); DICH.mkdir(exist_ok=True)
HAN = re.compile(r"[㐀-鿿\U00020000-\U0002ffff]")

def split_long(line, n):
    """Cắt dòng dài hơn n chữ ở dấu câu (không có dấu thì cắt cứng)."""
    parts, cur = [], ""
    for s in re.split(r"(?<=[。！？；!?;])", line):
        while len(s) > n:
            if cur:
                parts.append(cur); cur = ""
            parts.append(s[:n]); s = s[n:]
        if cur and len(cur) + len(s) > n:
            parts.append(cur); cur = ""
        cur += s
    return parts + ([cur] if cur else [])

def make_chunks(text, n):
    """Gom các dòng liên tiếp thành đoạn <= n chữ."""
    chunks, cur = [], []
    for line in (l.strip() for l in text.splitlines()):
        for piece in split_long(line, n) if line else []:
            if cur and sum(map(len, cur)) + len(piece) > n:
                chunks.append("\n".join(cur)); cur = []
            cur.append(piece)
    return chunks + (["\n".join(cur)] if cur else [])

books = {}
for p in sorted(glob.glob("/kaggle/input/**/*.txt", recursive=True)):
    b, text = Path(p).stem, open(p, encoding="utf-8", errors="ignore").read()
    if b in books or len(HAN.findall(text)) < 50:
        continue  # trùng tên / không phải văn bản chữ Hán
    books[b] = [{"id": f"{b}|{i}|{hashlib.md5(c.encode()).hexdigest()[:8]}", "zh": c}
                for i, c in enumerate(make_chunks(text, CHUNK_CHARS))]
    print(f"{b:<50} {len(text):8,} ký tự -> {len(books[b]):5d} đoạn")
assert books, "Không thấy file .txt chữ Hán nào trong /kaggle/input -- đã Add Input chưa?"
ZH = {c["id"]: c["zh"] for cs in books.values() for c in cs}
CHUNKS = str(WORK / "chunks.jsonl")
with open(CHUNKS, "w", encoding="utf-8") as f:
    for cs in books.values():
        f.writelines(json.dumps(c, ensure_ascii=False) + "\n" for c in cs)
json.dump({"system": SYSTEM, "user": USER}, open(WORK / "prompt.json", "w", encoding="utf-8"), ensure_ascii=False)

# Chạy tiếp lần trước: Add Input = Output của phiên trước -> đoạn đã dịch không dịch lại
for f in glob.glob("/kaggle/input/**/dich_raw/dich_shard*.jsonl", recursive=True):
    dst = RAW / (Path(f).stem + "_prev.jsonl")
    if not dst.exists():
        shutil.copy(f, dst); print("Nạp lại", f)

def run_dich(limit=None, budget_h=BUDGET_H):
    """N_GPU / GPU_PER_MODEL bản model chạy song song, chia đều các đoạn."""
    n = max(1, N_GPU // GPU_PER_MODEL)
    jobs = []
    for k in range(n):
        gpus = ",".join(str(g) for g in range(k * GPU_PER_MODEL, (k + 1) * GPU_PER_MODEL))
        cmd = [sys.executable, "/kaggle/working/dich_worker.py", "--model", MODEL, "--chunks", CHUNKS,
               "--prompt", str(WORK / "prompt.json"), "--out", str(RAW), "--shard", str(k), "--nshard", str(n),
               "--batch", str(BATCH), "--max-new", str(MAX_NEW_TOKENS), "--temperature", str(TEMPERATURE),
               "--rep-penalty", str(REPETITION_PENALTY)]
        cmd += (["--load-4bit"] if LOAD_4BIT else []) + (["--limit", str(limit)] if limit else [])
        log = RAW / f"_dich_{k}.log"
        jobs.append((log, subprocess.Popen(cmd, stdout=open(log, "a"), stderr=subprocess.STDOUT,
                                           env={**os.environ, "CUDA_VISIBLE_DEVICES": gpus})))
    t0 = t_print = time.time()
    while any(p.poll() is None for _, p in jobs):
        time.sleep(30)
        if time.time() - t0 > budget_h * 3600:
            print(f"!!! Hết {budget_h} giờ -> dừng, phần đã dịch vẫn được ghi ra"); [p.terminate() for _, p in jobs]; break
        if time.time() - t_print > 120:
            t_print = time.time()
            last = [(open(l, errors="ignore").read().strip().splitlines() or ["..."])[-3:] for l, _ in jobs]
            print(f"[{(time.time() - t0) / 60:5.1f} phút]\n" + "\n".join("\n".join(x) for x in last), flush=True)
    for log, p in jobs:
        if p.wait() not in (0, -15):
            print(f"\n!!! {log.name} lỗi, 20 dòng cuối log:")
            print("\n".join(open(log, errors="ignore").read().splitlines()[-20:]))

def load_dich():
    out = {}
    for f in RAW.glob("dich_shard*.jsonl"):
        for line in open(f, encoding="utf-8"):
            try:
                r = json.loads(line); out[r["id"]] = r
            except (json.JSONDecodeError, KeyError):
                pass
    return out

## 2. Dịch thử vài đoạn

In [ ]:
run_dich(limit=2, budget_h=1)
for r in list(load_dich().values())[:4]:
    print(f"ZH: {ZH.get(r['id'], '')[:200]}\nVI: {r['vi'][:400]}\n")

## 3. Dịch toàn bộ

In [ ]:
run_dich(limit=LIMIT)

## 4. Ghi ra txt

In [ ]:
done = load_dich()
for b, cs in books.items():
    vi = [done[c["id"]]["vi"] if c["id"] in done else "[chưa dịch]" for c in cs]
    (DICH / f"{b}.txt").write_text("\n\n".join(vi) + "\n", encoding="utf-8")
    n_ok = sum(c["id"] in done for c in cs)
    n_max = sum(done[c["id"]].get("hit_max", False) for c in cs if c["id"] in done)
    print(f"{b:<50} {n_ok}/{len(cs)} đoạn" + (f", {n_max} đoạn chạm MAX_NEW_TOKENS (có thể bị cụt)" if n_max else ""))
shutil.make_archive(str(WORK / "dich"), "zip", DICH)
print("\nKết quả: /kaggle/working/dich/*.txt và dich.zip")